<a href="https://colab.research.google.com/github/marcoastupp/feb-assets-pipeline/blob/main/FEB_Upscaler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# 1. INSTALAÇÃO LEVE SEM COMPILAÇÃO (SPANDREL + PYTORCH NATIVO)
# ==============================================================================
print("⚙️ Instalando motor moderno de Upscale (Spandrel)...")
!pip install -q spandrel opencv-python-headless pillow tqdm

# ==============================================================================
# 2. MONTAGEM DO GOOGLE DRIVE (ACEITE O POP-UP DE PERMISSÃO)
# ==============================================================================
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PASTA_ENTRADA = '/content/drive/MyDrive/FEB_Assets/01_Base_Flux'
PASTA_SAIDA   = '/content/drive/MyDrive/FEB_Assets/02_Upscaled_4x'

os.makedirs(PASTA_ENTRADA, exist_ok=True)
os.makedirs(PASTA_SAIDA, exist_ok=True)

# ==============================================================================
# 3. DOWNLOAD DO MODELO E EXECUÇÃO DO UPSCALE
# ==============================================================================
import urllib.request
import glob
import cv2
import torch
import numpy as np
from spandrel import ModelLoader
from tqdm import tqdm

weights_dir = '/content/weights'
os.makedirs(weights_dir, exist_ok=True)
model_path = os.path.join(weights_dir, 'RealESRGAN_x4plus.pth')

if not os.path.exists(model_path):
    print("⬇️ Baixando pesos do RealESRGAN_x4plus...")
    urllib.request.urlretrieve(
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
        model_path
    )

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ModelLoader().load_from_file(model_path).to(device).eval()

# Defina o fator final desejado: 2 (para ~2K) ou 4 (para ~4K)
FATOR_FINAL = 2

def upscale_image(img_bgr, scale_out=2, tile_size=512, overlap=32):
    # Converte BGR -> RGB Tensor [1, 3, H, W] em FP16 (rápido na T4)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    tensor = torch.from_numpy(img_rgb).permute(2, 0, 1).unsqueeze(0).to(device).half()

    _, _, h, w = tensor.shape
    scale_model = model.scale
    output = torch.zeros((1, 3, h * scale_model, w * scale_model), dtype=torch.half, device=device)

    with torch.no_grad(), torch.amp.autocast('cuda'):
        for y in range(0, h, tile_size - overlap):
            for x in range(0, w, tile_size - overlap):
                y2 = min(y + tile_size, h)
                x2 = min(x + tile_size, w)
                tile = tensor[:, :, y:y2, x:x2]
                out_tile = model(tile)
                output[:, :, y*scale_model:y2*scale_model, x*scale_model:x2*scale_model] = out_tile

    res = output.squeeze(0).permute(1, 2, 0).float().cpu().numpy()
    res = np.clip(res * 255.0, 0, 255).astype(np.uint8)
    res_bgr = cv2.cvtColor(res, cv2.COLOR_RGB2BGR)

    # Redimensiona se o fator final for menor que 4x (ex: 2x)
    if scale_out != scale_model:
        new_w = int(w * scale_out)
        new_h = int(h * scale_out)
        res_bgr = cv2.resize(res_bgr, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return res_bgr

extensoes = ('*.png', '*.jpg', '*.jpeg', '*.webp')
imagens = []
for ext in extensoes:
    imagens.extend(glob.glob(os.path.join(PASTA_ENTRADA, ext)))

if not imagens:
    print(f"⚠️ Nenhuma imagem encontrada em {PASTA_ENTRADA}. Coloque suas imagens lá e rode novamente!")
else:
    print(f"🚀 Processando {len(imagens)} imagem(ns) com fator {FATOR_FINAL}x...")
    for img_path in tqdm(imagens):
        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img is None:
            continue
        out_img = upscale_image(img, scale_out=FATOR_FINAL)
        nome_arquivo = os.path.basename(img_path)
        cv2.imwrite(os.path.join(PASTA_SAIDA, nome_arquivo), out_img)
    print(f"✨ Concluído! Imagens salvas em: {PASTA_SAIDA}")

⚙️ Instalando dependências do Real-ESRGAN...
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'basicsr' when getting requirements to build wheel


MessageError: Error: credential propagation was unsuccessful